# FLUX.2 Klein-4B on Kaggle TPU

Text-to-image generation, running natively in JAX on a TPU v5e-8.

**Before you run:** open the sidebar and check two things.
- **Accelerator** is set to **TPU VM v5e-8**
- The weights dataset is **attached** under Input

Then use **Run All** and wait. Nothing below needs editing.

The first run compiles the model, which takes a few minutes. Every run
after that reuses the compiled result from `/kaggle/working`, which
Kaggle keeps between sessions, so it starts in seconds.

**No safety filtering.** This includes no content moderation or output
filtering of any kind. You are responsible for what you generate.

## 1. Get the code

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/JohanesSetiawan/flux2-tpu.git"
REPOSITORY_DIRECTORY = Path("/kaggle/working/flux2-tpu")

# Safe to re-run, and safe to run alone after a kernel restart. A restart
# resets the working directory and clears sys.path, so a cell that only
# changed directory the first time would leave the package unimportable.
if REPOSITORY_DIRECTORY.exists():
    subprocess.run(["git", "-C", str(REPOSITORY_DIRECTORY), "pull", "--quiet"], check=False)
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPOSITORY_DIRECTORY)],
        check=True,
    )

os.chdir(REPOSITORY_DIRECTORY)
if str(REPOSITORY_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_DIRECTORY))

import src  # noqa: F401  Fail here, loudly, rather than three cells later.

print("code ready")

## 2. Install dependencies

In [ ]:
%pip install --quiet orbax-checkpoint tokenizers jinja2 gradio pillow

## 3. Find the weights

The weights come from an attached Kaggle dataset rather than a download.
That is the difference between starting in ten seconds and starting in
two minutes.

This cell **stops with an error** if the dataset is not attached, rather
than quietly falling back to downloading eleven gigabytes. A silent
fallback is how you end up waiting two minutes every run without
realising the dataset was never being used.

In [ ]:
from pathlib import Path

WEIGHTS_SEARCH_ROOTS = (
    Path("/kaggle/input/datasets/johaness14/flux-2-4b-klein-jax"),
    Path("/kaggle/input/flux-2-4b-klein-jax"),
)

REQUIRED_ENTRIES = ("params", "tokenizer", "manifest.json")


def locate_weights() -> Path:
    """Find the attached dataset, checking both mount layouts Kaggle uses."""
    for candidate in WEIGHTS_SEARCH_ROOTS:
        if all((candidate / entry).exists() for entry in REQUIRED_ENTRIES):
            return candidate

    attached = sorted(p.name for p in Path("/kaggle/input").glob("*")) or ["(none)"]
    raise FileNotFoundError(
        "The weights dataset is not attached, or does not have the expected "
        f"layout {REQUIRED_ENTRIES}.\n"
        f"Looked in: {[str(p) for p in WEIGHTS_SEARCH_ROOTS]}\n"
        f"Found under /kaggle/input: {attached}\n"
        "Open the sidebar, choose Input, then Add Input, and attach the "
        "weights dataset."
    )


WEIGHTS_DIRECTORY = locate_weights()
print(f"weights: {WEIGHTS_DIRECTORY}")

# The fast tokenizer needs the full pipeline definition. Without it the
# loader still works but falls back to a much slower path, so it is
# worth knowing which one you are on.
has_definition = (WEIGHTS_DIRECTORY / "tokenizer" / "tokenizer.json").is_file()
print("fast tokenizer:", "yes" if has_definition else "no, will use the slower fallback")

## 4. Configure

Nothing here needs changing. The compilation cache lives in
`/kaggle/working`, which Kaggle keeps between sessions, so the minutes
spent compiling on the first run are not spent again.

In [ ]:
import logging
from pathlib import Path

from src.config import (
    CheckpointSourceConfig,
    ExecutionConfig,
    InferenceConfig,
    MemoryResidencyStrategy,
)
from src.utils import configure_logging

WORKING_DIRECTORY = Path("/kaggle/working")

logger = configure_logging(
    log_file_path=WORKING_DIRECTORY / "generation_log.txt",
    logger_name="flux2_klein",
)

inference_config = InferenceConfig(
    checkpoint_source=CheckpointSourceConfig(
        # Pointing at the attached dataset means nothing is downloaded.
        local_cache_directory=WEIGHTS_DIRECTORY,
    ),
    # AUTO picks from the visible device count. On a v5e-8 that is
    # fully resident: every component stays in accelerator memory.
    residency_strategy=MemoryResidencyStrategy.AUTO,
)

execution_config = ExecutionConfig(
    compilation_cache_directory=WORKING_DIRECTORY / "compilation_cache",
)

print("configured")

## 5. Load and prepare

Restores the three model components and compiles the generation program
for each supported resolution.

On a cold compilation cache this takes a few minutes, almost all of it
compiling rather than loading. On later runs the cache is reused and
this finishes in seconds.

In [ ]:
from src.pipeline import Pipeline

pipeline = Pipeline(inference_config, logger, execution_config=execution_config)
pipeline.load()
pipeline.warm_up()

print("ready to generate")

## 6. Generate

Type a prompt and press Generate. Repeat as often as you like, with the
same prompt or different ones; nothing needs restarting between images.

A seed of `-1` picks a random one. The seed actually used is shown after
each image, so anything you like can be reproduced by entering that seed
again.

Reusing a prompt is faster than changing it, because the text encoding
is cached.

In [ ]:
from src.interfaces.browser import build_interface

# share=True gives a public link, which is how a Kaggle notebook is
# normally reached from a browser. The link is live only while this cell
# runs.
build_interface(pipeline, logger).launch(share=True)

## Notes

**After a kernel restart**, run cell 1 again before anything else. It
will not re-clone or re-download; it only restores the working directory
and the import path, which a restart clears.

**Three resolutions are offered**, and that is deliberate. Past roughly
4300 image tokens the reference sampling schedule switches to a formula
derived for a fifty-step model, and this checkpoint takes four steps.

**The full log** is written to `/kaggle/working/generation_log.txt`,
including a timing breakdown of every stage. It survives the notebook
output being cleared.